In [ ]:
! pip install "pyautogen>=0.2.18" "tavily-python"

In [ ]:
import os
from typing import Annotated
from tavily import TavilyClient
from autogen import AssistantAgent, UserProxyAgent, config_list_from_json, register_function
from dotenv import load_dotenv

load_dotenv(override=True)

In [ ]:
llm_config = {
    "model": "gpt-4o",
    "api_key": os.getenv("OPENAI_API_KEY")
}

## Tavily as a tool for searching the web

In [ ]:
tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

def search_tool(query: Annotated[str, "The Search query"]) -> Annotated[str, "The search results"]:
    return tavily.get_search_context(query=query, search_depth="advanced")

## Adding a ReAct promot

In [ ]:
ReAct_prompt = """
Answer the following questions as best you can. You have access to tools provided.

Use the following format:

Question: The input question you must answer
Thought: you should always think about what to do
Action: the action to take
Action Input: the input to the action
Observation: the result of the action
... (this  process can repeat multiple times)
Thought: I now know the final answer
Final Answer: The final answer to the original input question

Begin!
Question: {input}
"""

def react_prompt_message(sender, recipient, context):
    return ReAct_prompt.format(input=context["question"])

## Define agents

In [ ]:
user_proxy = UserProxyAgent(
    name="User",
    is_termination_msg=lambda x: x.get("content", "") and x.get("content", "").rstrip().endswith("TERMINATE"),
    human_input_mode="ALWAYS",
    max_consecutive_auto_reply=10,
    code_execution_config=False
)

research_assistant = AssistantAgent(
    name="Assistant",
    system_message="""You are a helpful research assistant who has the ability to search the web using the provided tools.
    Only use the tools you have been provided with. Reply TERMINATE when the task is done.""",
    llm_config=llm_config,
)


## Register the search tool

In [ ]:
register_function(
    search_tool,
    caller=research_assistant,
    executor=user_proxy,
    name="search_tool",
    description="Search the web for the given query",
)

## Initiate the chat

In [ ]:
user_proxy.initiate_chat(
    research_assistant,
    prompt_message=react_prompt_message,
    message= react_prompt_message,
    question=".........Write your question here.........",
)